# Part 3: Sales Forecasting Model — DATATHON 2026
**Team:** GenCore

**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01.

**Strategy:** Seasonal profile (day-of-year) + YoY growth trend.
**Model:** XGBoost/LightGBM integrating `web_traffic` features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.preprocessing import load_and_merge_order_data
from src.utils import calculate_metrics

RANDOM_SEED = 42
VND_RATE = 25450
DATA_PATH = '../data/raw/'

## 1 — Data Loading & Preprocessing
We use the `sales.csv` for historical daily Revenue/COGS in VND.

In [ ]:
sales = pd.read_csv(os.path.join(DATA_PATH, 'sales.csv'), parse_dates=['Date'])
web_traffic = pd.read_csv(os.path.join(DATA_PATH, 'web_traffic.csv'), parse_dates=['date'])

# Ensure currency is in VND (if not already handled in raw)
# Reference rate: 1 USD approx 25,450 VND

## 2 — Feature Engineering
**Note:** Strictly avoiding data leakage. We do not use Revenue/COGS of the test set as features.

In [ ]:
def create_features(df, traffic_df=None):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['dayofyear'] = df['Date'].dt.dayofyear
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    if traffic_df is not None:
        # Merge web traffic features (sessions, conversion_rate)
        df = df.merge(traffic_df[['date', 'sessions', 'conversion_rate']], 
                      left_on='Date', right_on='date', how='left')
        df.drop(columns=['date'], inplace=True)
    
    return df

## 3 — Modeling: Seasonal Profile + Trend

In [ ]:
# Implementation of Seasonal Profile + YoY growth trend scale
pass